In [1]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
import os
import psycopg2
import psycopg2.pool
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv
import datetime
from pydantic import BaseModel, Field
from typing import Optional
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

In [2]:
load_dotenv()

True

In [3]:
DATABASE_URL = os.getenv("DATABASE_URL")
SENDER_EMAIL= os.getenv("SENDER_EMAIL")
PASSWORD= os.getenv("PASSWORD")
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")

In [4]:
pool = psycopg2.pool.SimpleConnectionPool(
    2, 
    3, 
    dsn=DATABASE_URL
)

print("Connection pool created successfully using DATABASE_URL")

Connection pool created successfully using DATABASE_URL


In [ ]:
def send_confirmation_email(recipient_email, name, date, time, reason):
    sender_email = SENDER_EMAIL
    password = PASSWORD 
    
    message = MIMEMultipart()
    message["From"] = sender_email
    message["To"] = recipient_email
    message["Subject"] = "Appointment Confirmation - Health Clinic"

    body = f"""
    <html>
        <body>
            <h2>Hello, {name}!</h2>
            <p>Your appointment has been successfully scheduled.</p>
            <hr>
            <p><strong>Date:</strong> {date}</p>
            <p><strong>Time:</strong> {time}</p>
            <p><strong>Reason:</strong> {reason}</p>
            <hr>
            <p>If you need to reschedule, please contact us 24 hours in advance.</p>
        </body>
    </html>
    """
    message.attach(MIMEText(body, "html"))

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(sender_email, password)
        server.sendmail(sender_email, recipient_email, message.as_string())

In [6]:
@tool
def get_current_date_time() -> str:
    """
    Returns the current date and time in a human-readable format 
    and an ISO format for database queries.
    """
    now = datetime.datetime.now()

    # Formatted Date: Friday, February 20, 2026
    readable_date = now.strftime("%A, %B %d, %Y")
    
    # Formatted Time: 01:32 PM
    readable_time = now.strftime("%I:%M %p")
    
    # ISO Format: 2026-02-20
    iso_date = now.date().isoformat()

    return (
        f"Today is {readable_date}. "
        f"The current time is {readable_time}. "
        f"Query format: {iso_date}"
    )

In [ ]:
@tool
def list_available_services():
    """
    List all available medical services, including their modality, 
    price, and duration in minutes.
    """
    conn = None
    try:
        conn = pool.getconn()
        
        with conn:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("""
                    SELECT name, price, duration_minutes, modality 
                    FROM services 
                    WHERE is_active = TRUE 
                    ORDER BY name ASC;
                """)
                rows = cur.fetchall()

        if not rows:
            return "No hay servicios disponibles actualmente."

        lines = [
            f"- {s['name']} ({s['modality']}): ${s['price']}, {s['duration_minutes']} min"
            for s in rows
        ]
        return "\n".join(lines)

    except Exception as e:
        print(f"❌ Database Error: {e}")
        return "Lo siento, hubo un error al consultar el catálogo de servicios."
        
    finally:
        if conn:
            pool.putconn(conn)

In [ ]:
@tool
def check_availability(date: str) -> str:
    """
    Checks the availability of appointments for a specific date (YYYY-MM-DD).
    Returns a list of occupied times or a confirmation that the day is free.
    """
    query = """
        SELECT appointment_time 
        FROM appointments 
        WHERE appointment_date = %s 
        AND status != 'Cancelled' 
        ORDER BY appointment_time ASC;
    """
    
    conn = None
    try:
        conn = pool.getconn()
        
        with conn.cursor() as cur:
            cur.execute(query, (date,))
            rows = cur.fetchall()
            
            if not rows:
                return f"Everything is available for {date}."
            
            # rows is a list of tuples, e.g., [(datetime.time(10, 0),), (datetime.time(11, 30),)]
            occupied = ", ".join([r[0].strftime("%H:%M") for r in rows])
            
            return f"On {date}, these slots are occupied: {occupied}."
            
    except Exception as e:
        print(f"Database error: {e}")
        return "Error checking availability."
    
    finally:
        if conn is not None:
            pool.putconn(conn)

In [9]:
class BookingArgs(BaseModel):
    full_name: str = Field(description="Patient's full name")
    phone: str = Field(description="Patient's phone number")
    email: Optional[str] = Field(None, description="Patient's email address")
    birth_date: str = Field(description="Birth date in YYYY-MM-DD format")
    age: int = Field(description="Patient's age")
    gender: str = Field(description="Patient's gender")
    service_id: int = Field(description="ID of the service being booked")
    appointment_date: str = Field(description="Date of appointment YYYY-MM-DD")
    appointment_time: str = Field(description="Time of appointment HH:MM")
    reason: Optional[str] = Field(None, description="Reason for the visit")

In [ ]:
@tool(args_schema=BookingArgs)
def book_appointment(
    full_name, phone, email, birth_date, age, gender, 
    service_id, appointment_date, appointment_time, reason
) -> str:
    """
    Registers a patient (if new) and books an appointment. 
    Checks for conflicts before finalizing.
    """
    conn = None
    try:
        conn = pool.getconn()
        conn.autocommit = False
        with conn.cursor() as cur:
            
            cur.execute("""
                INSERT INTO patients (full_name, phone, email, birth_date, age, gender) 
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT (phone) DO UPDATE SET full_name = EXCLUDED.full_name 
                RETURNING id;
            """, (full_name, phone, email, birth_date, age, gender))
            patient_id = cur.fetchone()[0]

            cur.execute("""
                SELECT id FROM appointments 
                WHERE appointment_date=%s AND appointment_time=%s AND status!='Cancelled'
            """, (appointment_date, appointment_time))
            
            if cur.fetchone():
                conn.rollback()
                return "This slot is already occupied."

            cur.execute("""
                INSERT INTO appointments (patient_id, service_id, appointment_date, appointment_time, status, reason)
                VALUES (%s, %s, %s, %s, 'Pending', %s) RETURNING id;
            """, (patient_id, service_id, appointment_date, appointment_time, reason))
            appointment_id = cur.fetchone()[0]

            conn.commit()
            try:
                send_confirmation_email(email, full_name, appointment_date, appointment_time, reason)
                email_status = "Confirmation email sent."
            except Exception as e:
                print(f"Email failed: {e}")
                
            return f"Appointment successfully scheduled! ID: {appointment_id}."

    except Exception as e:
        if conn:
            conn.rollback()
        print(f"❌ Error booking: {e}")
        return "Internal error during booking."
    
    finally:
        if conn:
            pool.putconn(conn)

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [12]:
agent = create_react_agent(
    model=llm,
    tools=[list_available_services,get_current_date_time,check_availability,book_appointment],
    prompt="You are a helpful assistant", 
)

/var/folders/7b/7jsdpl9s5lz119fc92f8g5k40000gn/T/ipykernel_1594/364370047.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
history = []

print("Chatbot started! (Type 'quit' or 'exit' to stop)")

while True:
    user_query = input("You: ")
    
    if user_query.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break

    history.append(("user", user_query))
    
    inputs = {"messages": history}
    result = agent.invoke(inputs)
    
    ai_response = result["messages"][-1]
    print(f"AI: {ai_response.content}")
    
    history.append(ai_response)

Chatbot started! (Type 'quit' or 'exit' to stop)
AI: I can help you with that! To book an appointment, I need a few details from you:

1. Your full name
2. Your phone number
3. Your email address (optional)
4. Your birth date (in YYYY-MM-DD format)
5. Your age
6. Your gender
7. The reason for the visit (if you have one)

Also, let me know if you have a preferred date and time for the appointment. If not, I can check for availability.
AI: Here are the available medical services:

1. **Automated Follow-up Care (Virtual)**
   - Price: $15.00
   - Duration: 5 minutes

2. **COVID-19 Initial Triage (Virtual)**
   - Price: $0.00
   - Duration: 15 minutes

3. **Home PCR Collection (Onsite)**
   - Price: $120.00
   - Duration: 30 minutes

4. **In-Clinic Antigen Test (Onsite)**
   - Price: $30.00
   - Duration: 15 minutes

5. **Specialist Teleconsultation (Virtual)**
   - Price: $55.00
   - Duration: 20 minutes

Please let me know which service you would like to book, as well as the other detail

In [ ]:
''' 
1-Hello, I feel sick and I would like an appointment

2-What services do you offer?

3-I would like to schedule an appointment for Home PCR Collection

4-Today at 5pm

5-Charly Mendez, 6638519572, 2002-01-25, 24, Male, charly1350@gmail.com, Home PCR Collection

5-Charly Mendez, 6638519572, 2002-01-25, 2026-02-20, 17:00, 24, Male, charly1350@gmail.com, Home PCR Collection


'''